# Sentence-level retrieval cho corpus

## Goal

Đọc các câu đã word-segmented từ PostgreSQL, tokenization theo batch bằng
`vinai/phobert-base`, mean pooling thành vector 768 chiều và upsert vào pgvector.

Notebook mặc định chỉ xử lý bài đầu tiên để `Run All` luôn bounded. CLI `make embed` mới là
entrypoint chạy toàn corpus.

## Setup

Chạy notebook từ thư mục `backend/`.

In [1]:
from information_retrieval.application.embed_segmented_sentences import (
    EmbedSegmentedSentences,
)
from information_retrieval.infrastructure.config import get_settings
from information_retrieval.infrastructure.database import create_database_engine
from information_retrieval.infrastructure.phobert_sentence_encoder import (
    PhoBertSentenceEncoder,
)
from information_retrieval.infrastructure.sentence_embedding_repository import (
    PostgresSentenceEmbeddingRepository,
)

/Users/thangtran/Workplace/master_s_degree/information_retrieval/.worktrees/sentence-embedding-pipeline/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data

`CRAWL_ID=None` ở bước chọn mẫu nghĩa là lấy bài đầu tiên hiện có, không phải chạy toàn corpus.

In [2]:
CRAWL_ID = None

settings = get_settings()
engine = create_database_engine(settings.database_url)
repository = PostgresSentenceEmbeddingRepository(engine)
repository.initialize_schema()

available_sentences = repository.list_for_embedding(CRAWL_ID)
if not available_sentences:
    raise RuntimeError("Database chưa có segmented_sentences; hãy chạy make segment trước.")

effective_crawl_id = CRAWL_ID or available_sentences[0].crawl_url_id
document_sentences = repository.list_for_embedding(effective_crawl_id)

print(f"crawl_url_id={effective_crawl_id}")
print(f"sentences={len(document_sentences)}")
print(document_sentences[0].segmented_text)

crawl_url_id=1
sentences=20
Đại_biểu muốn gọi xe cấp_cứu thuận_tiện như gọi taxi


## Steps

### 1. Khởi tạo PhoBERT encoder

In [3]:
encoder = PhoBertSentenceEncoder(
    model_name=settings.phobert_model_name,
    cache_dir=settings.phobert_model_dir,
    max_length=settings.embedding_max_length,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 63414.87it/s]


[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 2. Tokenization, pooling và upsert một bài

In [4]:
summary = EmbedSegmentedSentences(
    sentence_source=repository,
    encoder=encoder,
    embedding_repository=repository,
    model_name=settings.phobert_model_name,
    batch_size=settings.embedding_batch_size,
).execute(effective_crawl_id)

print(summary)

EmbeddingSummary(selected_documents=1, embedded_documents=1, selected_sentences=20, stored_embeddings=20, failures=[])


## Checks

In [5]:
assert not summary.failures
assert summary.selected_documents == 1
assert summary.selected_sentences == len(document_sentences)
assert summary.stored_embeddings == len(document_sentences)

print("Sentence embedding smoke check passed.")

Sentence embedding smoke check passed.


## Next Steps

- Chạy một bài từ terminal: `make embed CRAWL_ID=<id>`.
- Chạy toàn corpus: `make embed`.
- Bước tiếp theo là encode query bằng cùng model và word-segmentation policy, sau đó dùng
  cosine distance của pgvector để xếp hạng câu.